In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from utils import aproximar_limite_valido

In [3]:
df = pd.read_csv("../data/comprovantes_pix_10000_anomalias.csv", sep=";")

In [4]:
df['DataHora_Tratada'] = df['DataHora'].apply(aproximar_limite_valido)

In [5]:
# O .dt.hour e .dt.dayofweek roubam apenas a hora e o dia da semana (onde 0 é segunda e 6 é domingo).
# função lambda para dizer: Se o dia for 5 (sábado) ou 6 (domingo), marque 1 (É fim de semana). Senão, marque 0.
df['Hora'] = df['DataHora_Tratada'].dt.hour
df['DiaDaSemana'] = df['DataHora_Tratada'].dt.dayofweek
df['FimDeSemana'] = df['DiaDaSemana'].apply(lambda x: 1 if x >= 5 else 0)

# Horário comercial ganha 1 apenas se for entre 8h e 18h e se não for fim de semana. Já a madrugada ganha 1 se a hora for de 0 a 5.
df['Horario_Comercial'] = df.apply(lambda row: 1 if (8 <= row['Hora'] <= 18) and (row['FimDeSemana'] == 0) else 0, axis=1)
df['Madrugada'] = df['Hora'].apply(lambda x: 1 if 0 <= x <= 5 else 0)

# Isola os dias de pico bancário (salários e vales). 
df['Dia_do_Mes'] = df['DataHora_Tratada'].dt.day
df['Dia_de_Pagamento'] = df['Dia_do_Mes'].apply(lambda x: 1 if x in [5, 6, 7, 20, 21, 22] else 0)

# Faz uma divisão por 1 (usando o %). Se o resto for zero, significa que não tem centavos (ex: R$ 500,00 redondo). 
df['Valor_Redondo'] = df['Valor'].apply(lambda x: 1 if x % 1 == 0 else 0)

In [6]:
df.agg(['max', 'min'])

,EndToEndId,DataHora,Valor,Moeda,Pagador_Nome,Pagador_CPF_CNPJ,Pagador_Banco,Recebedor_Nome,Recebedor_CPF_CNPJ,Recebedor_Banco,...,Anomalia,DataHora_Tratada,Hora,DiaDaSemana,FimDeSemana,Horario_Comercial,Madrugada,Dia_do_Mes,Dia_de_Pagamento,Valor_Redondo
max,ffffad37-8e79-4cd8-a1b6-b05d20bc5b5e,2025-08-20 15:44:56,4999.79,BRL,Yuri da Rosa,999.790.314-14,Santander Brasil,Yuri das Neves,999.704.956-96,Santander Brasil,...,1,2025-08-20 15:44:56,23,6,1,1,1,31,1,1
min,00032a00-5360-40da-aa43-ea59ce48593e,2025-02-30 25:61:00,0.00,BRL,Agatha Almeida,000.000.000-00,BTG Pactual,Agatha Araújo,10.127.878/0001-59,BTG Pactual,...,0,2025-02-28 23:59:00,0,0,0,0,0,1,0,0


In [51]:
df_filtrado = df[
    (df['Anomalia'] == 0) 
    & (df['Valor'] <= 2650) 
    & ((df['Dia_do_Mes'] >= 25) | (df['Dia_do_Mes'] <= 3))
    & (df['Hora'] >= 18) 
]

In [53]:
lista_400_valores = df_filtrado['EndToEndId'].head(400).tolist()

In [54]:
df01 = pd.read_csv("../data/comprovantes_pix_10000_anomalias.csv", sep=";")

In [56]:
df01.loc[df01['EndToEndId'].isin(lista_400_valores), 'Anomalia'] = 1

In [57]:
df01.value_counts(subset=['Anomalia'])

Anomalia
0           9500
1            500
Name: count, dtype: int64

In [58]:
df01.to_csv("../data/base_bruta.csv", index=False)